# F1 Strategy Dataset Builder
Collects data from OpenF1 and FastF1, saves to CSV for model training.
Run this once to build the dataset, then use 2_train_model.py for training.

In [1]:
!pip install fastf1

import re
import math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import fastf1
from fastf1.core import Session, exceptions

pd.set_option('display.max_columns', 200)
np.random.seed(42)

# --------------------
# Configuration
# --------------------
DATA_DIR = Path("openf1_data")
assert DATA_DIR.exists(), f"Missing: {DATA_DIR.resolve()}"

# FastF1 cache (recommended)
FASTF1_CACHE_DIR = Path("fastf1_cache")
FASTF1_CACHE_DIR.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(FASTF1_CACHE_DIR))

# Output files
OUTPUT_DATASET = "f1_strategy_dataset.csv"
OUTPUT_WEATHER_SEQS = "f1_weather_sequences.npz"

WEATHER_FEATURES = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
DRY_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]
DEFAULT_PIT_LOSS = 22.0  # seconds

# FastF1 collection range
FASTF1_START_YEAR = 2018
FASTF1_END_YEAR = datetime.utcnow().year
FASTF1_SESSION_NAME = "R"  # Race
FASTF1_MAX_SESSIONS = None  # Set to a number like 5 to limit for testing


# --------------------
# Utility Functions
# --------------------
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None

def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def safe_mode(series, default="MEDIUM"):
    s = pd.Series(series).dropna()
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else default

def ensure_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in WEATHER_FEATURES:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def compute_weather_seq_and_summary(weather_df: pd.DataFrame):
    if weather_df is None or len(weather_df) == 0:
        return None, None
    w = ensure_weather_columns(weather_df)
    w = w.dropna(subset=WEATHER_FEATURES, how="all")
    if w.empty:
        return None, None

    seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
    rain = w["rainfall"].fillna(0)
    temp_delta = w["track_temperature"] - w["air_temperature"]
    rain_change_rate = float(rain.diff().abs().mean()) if len(rain) > 1 else 0.0

    summary = {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "air_temp_std": float(w["air_temperature"].std()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "track_temp_std": float(w["track_temperature"].std()),
        "humidity_mean": float(w["humidity"].mean()),
        "humidity_std": float(w["humidity"].std()),
        "rain_minutes_ratio": float((rain > 0).mean()),
        "rain_rate_mean": float(rain.mean()),
        "rain_rate_std": float(rain.std()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
        "wind_speed_std": float(w["wind_speed"].std()),
        "temp_delta_mean": float(temp_delta.mean()),
        "rain_change_rate": float(rain_change_rate),
    }
    return seq, summary

def normalize_compound(name: str, default="MEDIUM"):
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return default
    s = str(name).upper().strip()
    if s not in VALID_COMPOUNDS:
        return default
    return s

def estimate_pit_loss_from_fastf1_pits(pit_laps_df: pd.DataFrame):
    """Estimate total pit time loss from FastF1."""
    if pit_laps_df is None or len(pit_laps_df) == 0:
        return 0.0

    pits = pit_laps_df[pit_laps_df.get("PitInTime").notna()]
    if len(pits) == 0:
        return 0.0

    if "PitInTime" in pits.columns and "PitOutTime" in pits.columns:
        try:
            delta = (pits["PitOutTime"] - pits["PitInTime"]).dt.total_seconds()
            delta = pd.to_numeric(delta, errors="coerce").dropna()
            if len(delta) > 0:
                delta = delta.clip(lower=0)
                return float(delta.sum())
        except Exception:
            pass

    return float(DEFAULT_PIT_LOSS * len(pits))

def compute_openf1_stint_stats(team_stints: pd.DataFrame):
    lengths = pd.Series(dtype=float)
    if "stint_length" in team_stints.columns:
        lengths = pd.to_numeric(team_stints["stint_length"], errors="coerce")
    elif "lap_start" in team_stints.columns and "lap_end" in team_stints.columns:
        lap_start = pd.to_numeric(team_stints["lap_start"], errors="coerce")
        lap_end = pd.to_numeric(team_stints["lap_end"], errors="coerce")
        lengths = (lap_end - lap_start + 1)

    lengths = lengths.dropna()
    num_stints = float(team_stints["stint_number"].max()) if "stint_number" in team_stints.columns else np.nan

    if len(lengths) == 0:
        return {
            "total_stint_laps": np.nan,
            "avg_stint_laps": np.nan,
            "max_stint_laps": np.nan,
            "num_stints": num_stints,
        }

    return {
        "total_stint_laps": float(lengths.sum()),
        "avg_stint_laps": float(lengths.mean()),
        "max_stint_laps": float(lengths.max()),
        "num_stints": num_stints,
    }

def compute_fastf1_lap_stats(team_laps: pd.DataFrame):
    lap_time_sec = None
    if "LapTime" in team_laps.columns:
        lap_time_sec = pd.to_timedelta(team_laps["LapTime"], errors="coerce").dt.total_seconds()
        lap_time_sec = pd.to_numeric(lap_time_sec, errors="coerce")

    stint_lengths = None
    if "Stint" in team_laps.columns:
        stint_lengths = team_laps.groupby("Stint")["LapNumber"].count()

    tyre_life = pd.to_numeric(team_laps.get("TyreLife"), errors="coerce") if "TyreLife" in team_laps.columns else None

    non_green_ratio = np.nan
    if "TrackStatus" in team_laps.columns:
        status = team_laps["TrackStatus"].astype(str)
        non_green_ratio = float((status != "1").mean()) if len(status) else np.nan

    pace_trend = 0.0
    if lap_time_sec is not None and "LapNumber" in team_laps.columns:
        lt = pd.to_numeric(lap_time_sec, errors="coerce").dropna()
        lap_num = pd.to_numeric(team_laps.loc[lt.index, "LapNumber"], errors="coerce").dropna()
        if len(lt) >= 3 and len(lap_num) == len(lt):
            try:
                pace_trend = float(np.polyfit(lap_num.values, lt.values, 1)[0])
            except Exception:
                pace_trend = 0.0

    return {
        "total_laps": float(team_laps["LapNumber"].max()) if "LapNumber" in team_laps.columns else np.nan,
        "avg_stint_laps": float(stint_lengths.mean()) if stint_lengths is not None and len(stint_lengths) else np.nan,
        "max_stint_laps": float(stint_lengths.max()) if stint_lengths is not None and len(stint_lengths) else np.nan,
        "mean_tyre_life": float(tyre_life.mean()) if tyre_life is not None and len(tyre_life.dropna()) else np.nan,
        "max_tyre_life": float(tyre_life.max()) if tyre_life is not None and len(tyre_life.dropna()) else np.nan,
        "mean_lap_time_sec": float(lap_time_sec.mean()) if lap_time_sec is not None and len(lap_time_sec.dropna()) else np.nan,
        "lap_time_std_sec": float(lap_time_sec.std()) if lap_time_sec is not None and len(lap_time_sec.dropna()) else np.nan,
        "pace_trend_per_lap": float(pace_trend),
        "non_green_lap_ratio": non_green_ratio,
    }


# --------------------
# OpenF1 Data Loading
# --------------------
def load_session_maps_openf1():
    sessions = safe_read_csv(DATA_DIR / "sessions.csv")
    if sessions.empty:
        return {}, {}, {}

    for c in ["session_key", "meeting_key", "session_name", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan

    sessions["track_name"] = (
        sessions["circuit_short_name"].fillna(sessions["location"]).fillna(sessions["meeting_key"].astype(str))
    )

    return (
        dict(zip(sessions["session_key"], sessions["track_name"])),
        dict(zip(sessions["session_key"], sessions["meeting_key"])),
        dict(zip(sessions["session_key"], sessions["session_name"].astype(str))),
    )

def get_weather_seq_and_summary_openf1(session_key):
    w = safe_read_csv(DATA_DIR / f"weather_session_{session_key}.csv")
    if w.empty:
        return None, None
    w = ensure_weather_columns(w)
    return compute_weather_seq_and_summary(w)

def estimate_time_loss_target_openf1(session_key, team_name, driver_team):
    p = safe_read_csv(DATA_DIR / f"pit_session_{session_key}.csv")
    if p.empty or "driver_number" not in p.columns:
        return 0.0
    p = p.merge(driver_team, on="driver_number", how="left")
    p = p[p["team_name"] == team_name]
    if p.empty:
        return 0.0
    for c in ["pit_duration", "duration", "pit_time"]:
        if c in p.columns:
            d = pd.to_numeric(p[c], errors="coerce").dropna()
            if len(d):
                return float(d.sum() + 15.0 * len(d))
    return float(21.0 * len(p))

def build_samples_openf1(race_only=True):
    session_to_track_openf1, session_to_meeting_openf1, session_to_name_openf1 = load_session_maps_openf1()
    
    rows, seqs = [], []
    for sf in sorted(DATA_DIR.glob("stints_session_*.csv")):
        sid = parse_session_key(sf)
        if sid is None:
            continue
        sname = session_to_name_openf1.get(sid, "")
        if race_only and ("Race" not in sname and "RACE" not in sname):
            continue

        st = safe_read_csv(sf)
        dr = safe_read_csv(DATA_DIR / f"drivers_session_{sid}.csv")
        if st.empty or dr.empty:
            continue

        for c in ["driver_number", "compound", "stint_number", "starting_position", "lap_start", "lap_end", "stint_length"]:
            if c not in st.columns:
                st[c] = np.nan
        for c in ["driver_number", "team_name"]:
            if c not in dr.columns:
                dr[c] = np.nan

        st = st.merge(dr[["driver_number", "team_name"]], on="driver_number", how="left")
        st["compound"] = st["compound"].apply(normalize_compound)

        by_team = st.groupby("team_name", dropna=False)
        for tm, g in by_team:
            if pd.isna(tm):
                continue

            n_stops = int(g["stint_number"].max() - 1) if "stint_number" in g.columns else 0
            target_comp = safe_mode(g["compound"])
            start_comp = normalize_compound(g[g["stint_number"] == 1]["compound"].iloc[0] if len(g[g["stint_number"] == 1]) else None)
            start_pos = float(g["starting_position"].iloc[0]) if "starting_position" in g.columns else 10.0

            seq, ws = get_weather_seq_and_summary_openf1(sid)
            if seq is None or ws is None:
                continue

            time_loss = estimate_time_loss_target_openf1(sid, tm, dr)
            stint_stats = compute_openf1_stint_stats(g)

            rows.append({
                "source": "openf1",
                "session_key": sid,
                "team_name": str(tm),
                "track_name": session_to_track_openf1.get(sid, "Unknown"),
                "starting_position": start_pos,
                "starting_compound": start_comp,
                "stops": n_stops,
                "target_compound": target_comp,
                "time_loss_sec": time_loss,
                **ws,
                **stint_stats,
                # placeholders for lap/pace features not available in OpenF1 stints
                "total_laps": np.nan,
                "mean_tyre_life": np.nan,
                "max_tyre_life": np.nan,
                "mean_lap_time_sec": np.nan,
                "lap_time_std_sec": np.nan,
                "pace_trend_per_lap": np.nan,
                "non_green_lap_ratio": np.nan,
            })
            seqs.append(seq)

    df = pd.DataFrame(rows)
    return df, seqs


# --------------------
# FastF1 Data Collection
# --------------------
def fastf1_track_name(ev: pd.Series):
    for c in ["EventName", "Location", "OfficialEventName"]:
        if c in ev.index and pd.notna(ev[c]):
            return str(ev[c])
    return "Unknown"

def build_samples_fastf1():
    rows, seqs = [], []
    count = 0

    for year in range(FASTF1_START_YEAR, FASTF1_END_YEAR + 1):
        try:
            schedule = fastf1.get_event_schedule(year, include_testing=False)
        except Exception as e:
            print(f"Warning: Could not load schedule for {year}: {e}")
            continue

        for idx, event in schedule.iterrows():
            if FASTF1_MAX_SESSIONS is not None and count >= FASTF1_MAX_SESSIONS:
                print(f"Reached max sessions limit: {FASTF1_MAX_SESSIONS}")
                return pd.DataFrame(rows), seqs

            try:
                sess: Session = fastf1.get_session(year, event["RoundNumber"], FASTF1_SESSION_NAME)
                print(f"Loading: {year} {event['EventName']} - {FASTF1_SESSION_NAME}")
                sess.load(laps=True, telemetry=False, weather=True, messages=False)
            except Exception as e:
                print(f"  Error loading session: {e}")
                continue

            try:
                laps = sess.laps
            except exceptions.DataNotLoadedError:
                print("  Laps not loaded; skipping")
                continue

            if laps is None or len(laps) == 0:
                print("  No laps; skipping")
                continue

            # Weather
            w_df = sess.weather_data if hasattr(sess, "weather_data") else None
            if w_df is not None and len(w_df) > 0:
                w_df = w_df.copy()
                rename_map = {
                    "AirTemp": "air_temperature",
                    "TrackTemp": "track_temperature",
                    "Humidity": "humidity",
                    "Rainfall": "rainfall",
                    "WindSpeed": "wind_speed",
                }
                for src, dst in rename_map.items():
                    if src in w_df.columns and dst not in w_df.columns:
                        w_df[dst] = w_df[src]

                seq, ws = compute_weather_seq_and_summary(w_df)
            else:
                seq, ws = None, None

            if seq is None or ws is None:
                print("  No valid weather; skipping")
                continue

            track = fastf1_track_name(event)

            # By team
            by_team = laps.groupby("Team")
            for team_name, team_laps in by_team:
                if pd.isna(team_name) or len(team_laps) == 0:
                    continue

                # Stints
                stints = team_laps[["Stint", "Compound"]].drop_duplicates().sort_values("Stint")
                n_stops = len(stints) - 1
                target_comp = safe_mode(stints["Compound"])
                start_comp = normalize_compound(stints["Compound"].iloc[0] if len(stints) else None)

                # Starting position
                first_lap = team_laps[team_laps["LapNumber"] == 1]
                start_pos = float(first_lap["Position"].iloc[0]) if len(first_lap) and "Position" in first_lap.columns else 10.0

                # Pit loss
                pit_laps = team_laps[team_laps["PitInTime"].notna()]
                time_loss = estimate_pit_loss_from_fastf1_pits(pit_laps)

                # Lap/pace stats
                lap_stats = compute_fastf1_lap_stats(team_laps)

                rows.append({
                    "source": "fastf1",
                    "year": year,
                    "round": event["RoundNumber"],
                    "team_name": str(team_name),
                    "track_name": track,
                    "starting_position": start_pos,
                    "starting_compound": start_comp,
                    "stops": n_stops,
                    "target_compound": target_comp,
                    "time_loss_sec": time_loss,
                    **ws,
                    "total_stint_laps": lap_stats.get("total_laps"),
                    "avg_stint_laps": lap_stats.get("avg_stint_laps"),
                    "max_stint_laps": lap_stats.get("max_stint_laps"),
                    "num_stints": float(len(stints)),
                    **lap_stats,
                })
                seqs.append(seq)

            count += 1

    df = pd.DataFrame(rows)
    return df, seqs


# --------------------
# Main Execution
# --------------------
if __name__ == "__main__":
    print("="*60)
    print("Building F1 Strategy Dataset")
    print("="*60)
    
    print("\n[1/3] Building OpenF1 samples...")
    df_openf1, X_seq_openf1 = build_samples_openf1(race_only=True)
    print(f"  OpenF1 samples: {len(df_openf1)}")

    print("\n[2/3] Building FastF1 samples (this may take a while)...")
    df_fastf1, X_seq_fastf1 = build_samples_fastf1()
    print(f"  FastF1 samples: {len(df_fastf1)}")

    print("\n[3/3] Combining and saving...")
    df_combined = pd.concat([df_openf1, df_fastf1], ignore_index=True)
    X_seq_combined = X_seq_openf1 + X_seq_fastf1
    
    # Save dataset CSV
    df_combined.to_csv(OUTPUT_DATASET, index=False)
    print(f"  ✓ Saved dataset: {OUTPUT_DATASET} ({len(df_combined)} samples)")
    
    # Save weather sequences as numpy arrays
    # Store as list of arrays with different lengths
    np.savez_compressed(
        OUTPUT_WEATHER_SEQS,
        **{f"seq_{i}": seq for i, seq in enumerate(X_seq_combined)}
    )
    print(f"  ✓ Saved weather sequences: {OUTPUT_WEATHER_SEQS}")
    
    print("\n" + "="*60)
    print("Dataset building complete!")
    print("="*60)
    print(f"\nDataset shape: {df_combined.shape}")
    print(f"Weather sequences: {len(X_seq_combined)}")
    print(f"\nColumns: {list(df_combined.columns)}")
    print(f"\nSample counts by source:")
    print(df_combined['source'].value_counts())
    print(f"\nNext step: Run '2_train_model.py' to train the model")


  Using cached fastf1-3.8.3-py3-none-any.whl.metadata (5.1 kB)
  Using cached rapidfuzz-3.14.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (12 kB)
  Using cached requests_cache-1.3.2-py3-none-any.whl.metadata (9.4 kB)
  Using cached signalrcore-1.0.2-py3-none-any.whl.metadata (13 kB)
  Using cached timple-0.1.8-py3-none-any.whl.metadata (2.0 kB)
  Using cached websockets-16.0-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.8 kB)
  Using cached cattrs-26.1.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached url_normalize-3.0.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached msgpack-1.1.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached fastf1-3.8.3-py3-none-any.whl (135 kB)
Using cached requests_cache-1.3.2-py3-none-any.wh

/tmp/ipykernel_77653/1051171420.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  FASTF1_END_YEAR = datetime.utcnow().year


Building F1 Strategy Dataset

[1/3] Building OpenF1 samples...
  OpenF1 samples: 90

[2/3] Building FastF1 samples (this may take a while)...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Loading: 2018 Australian Grand Prix - R


...

  Error loading session: any API: 500 calls/h
Loading: 2024 Las Vegas Grand Prix - R


core           INFO 	Loading data for Qatar Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


  Error loading session: any API: 500 calls/h
Loading: 2024 Qatar Grand Prix - R


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


  Error loading session: any API: 500 calls/h
Loading: 2024 Abu Dhabi Grand Prix - R
  Error loading session: any API: 500 calls/h
  FastF1 samples: 1360

[3/3] Combining and saving...
  ✓ Saved dataset: f1_strategy_dataset.csv (1450 samples)
  ✓ Saved weather sequences: f1_weather_sequences.npz

Dataset building complete!

Dataset shape: (1450, 35)
Weather sequences: 1450

Columns: ['source', 'session_key', 'team_name', 'track_name', 'starting_position', 'starting_compound', 'stops', 'target_compound', 'time_loss_sec', 'air_temp_mean', 'air_temp_std', 'track_temp_mean', 'track_temp_std', 'humidity_mean', 'humidity_std', 'rain_minutes_ratio', 'rain_rate_mean', 'rain_rate_std', 'wind_speed_mean', 'wind_speed_std', 'temp_delta_mean', 'rain_change_rate', 'total_stint_laps', 'avg_stint_laps', 'max_stint_laps', 'num_stints', 'total_laps', 'mean_tyre_life', 'max_tyre_life', 'mean_lap_time_sec', 'lap_time_std_sec', 'pace_trend_per_lap', 'non_green_lap_ratio', 'year', 'round']

Sample counts b